# Structured Information Extraction with Groq's GPT-OSS Model

#### Developed By: Manaranjan Pradhan
#### www.manaranjanp.com

*This Jupyter notebook is confidential and proprietary to Manaranjan Pradhan. It is intended solely for authorized training purposes. Unauthorized distribution, sharing, or reproduction of this notebook or its contents is strictly prohibited. This material is for personal learning within the training program only and may not be used for commercial purposes or shared with others. Unauthorized use may result in disciplinary action or legal consequences. If you have received this notebook without authorization, please contact manaranjan@gmail.com immediately and delete all copies.*

---

## Overview

This notebook demonstrates how to extract **structured information** from unstructured HTML content using:

- **Groq's GPT-OSS model** (`openai/gpt-oss-120b`) — a powerful open-source language model served via the Groq API
- **Pydantic** — for defining strongly-typed schemas that validate the LLM's output
- **BeautifulSoup** — for parsing HTML content

### Why Structured Extraction?

Web pages contain rich information, but it's buried in HTML markup and natural language. Traditional parsing requires hand-crafted scrapers that break easily. By combining LLMs with **JSON schema enforcement**, we get reliable, type-safe data extraction with minimal code.

### What We Will Build

We'll extract a list of electric cars (model, price, range) from a ZigWheels webpage and turn it into a clean Pandas DataFrame.

### Pipeline

```
HTML file → BeautifulSoup (extract text)
        → Send to Groq GPT-OSS with Pydantic schema → Validated structured output → DataFrame
```

## Step 0: Install Required Libraries

We need:
- `groq` — official Groq Python SDK
- `pydantic` — for schema definition and validation
- `beautifulsoup4` — for HTML parsing
- `pandas` — for tabular display of results

In [2]:
!pip install -q groq pydantic beautifulsoup4 pandas


[notice] A new release of pip is available: 24.1.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Step 1: Read the HTML File and Extract Text

The webpage `zigwheels.html` contains an article listing popular electric cars in India. HTML is structured with tags, scripts, and styling — but for LLM extraction we only need the **visible text**.

We use **BeautifulSoup** to parse the HTML and call `.get_text()` to strip out all tags.

In [3]:
from bs4 import BeautifulSoup

# Open the HTML file and parse it with BeautifulSoup
with open("zigwheels.html") as fp:
    soup = BeautifulSoup(fp, 'html.parser')

# Extract all visible text from the parsed HTML
text = soup.get_text()

print(f"Total characters extracted: {len(text)}")

Total characters extracted: 5494


Let's preview a small chunk of the extracted text to understand what we're working with.

In [4]:
print(text[500:1000])

.99 Lakh

 315 km range ● 58 min charging time ● 45kW
View March Offers
Check On-Road Price »








4.1
|  63 reviews





Kia EV6	
Rs. 60.95 Lakh

 708 km range ● 18 min charging time ● 192 kmph
View March Offers
Check On-Road Price »








3.7
|  139 reviews





MG Comet EV	
Rs. 6.98 Lakh

 230 km range ● 3 hour 18 min charging time ● 41kW
View March Offers
Check On-Road Price »








5
|  read reviews





BYD Seal	
Rs. 41.00 Lakh

 650 km range
View March Offers
Check On-Road Price »



## Step 2: Configure the Groq API Client

Groq is a fast inference platform that hosts several open models, including OpenAI's `gpt-oss-120b`. To use it:

1. Get your API key from https://console.groq.com/keys
2. Set it as an environment variable named `GROQ_API_KEY`

The Groq client automatically reads from this environment variable.

In [5]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [6]:
from groq import Groq

# The client picks up GROQ_API_KEY from the environment automatically
client = Groq()

## Step 3: Define the Output Schema with Pydantic

**Pydantic** lets us declare the exact shape of the data we want back from the LLM. This gives us:

1. **Type safety** — fields have explicit types (`str`, `int`, `Optional[int]`, etc.)
2. **Field descriptions** — these are passed to the LLM as part of the JSON schema, helping it understand each field's meaning
3. **Validation** — Pydantic raises errors if the LLM returns malformed data
4. **Nested structures** — we can compose models for complex hierarchical outputs

### Our Schema

- `CarDetails` — represents a single EV (model name, price, range)
- `CarListResponse` — wraps a list of `CarDetails` (the LLM returns multiple cars)

We mark `range_kms` as `Optional[int]` because some cars in the source page don't list a range.

In [8]:
from typing import Optional, List
from pydantic import BaseModel, Field


class CarDetails(BaseModel):
    car_model: str = Field(description="The name of the EV model, e.g. 'Tata Nexon EV'")
    price: str = Field(description="Price of the model with unit, e.g. '14.49 Lakh' or '2.13 Crore'")
    range_kms: Optional[int] = Field(
        default=None,
        description="Kilometres the car can travel on a single full charge. None if not listed."
    )


class CarListResponse(BaseModel):
    cars: List[CarDetails] = Field(
        description="List of all electric vehicles mentioned in the input text"
    )

Let's inspect the JSON schema that Pydantic generates from our model. This is exactly what we'll send to Groq so it knows the desired output shape.

In [9]:
import json

print(json.dumps(CarListResponse.model_json_schema(), indent=2))

{
  "$defs": {
    "CarDetails": {
      "properties": {
        "car_model": {
          "description": "The name of the EV model, e.g. 'Tata Nexon EV'",
          "title": "Car Model",
          "type": "string"
        },
        "price": {
          "description": "Price of the model with unit, e.g. '14.49 Lakh' or '2.13 Crore'",
          "title": "Price",
          "type": "string"
        },
        "range_kms": {
          "anyOf": [
            {
              "type": "integer"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "Kilometres the car can travel on a single full charge. None if not listed.",
          "title": "Range Kms"
        }
      },
      "required": [
        "car_model",
        "price"
      ],
      "title": "CarDetails",
      "type": "object"
    }
  },
  "properties": {
    "cars": {
      "description": "List of all electric vehicles mentioned in the input text",


## Step 4: Call the GPT-OSS Model with JSON Schema Output

Now we make the actual API call. The key parameters:

- **`model="openai/gpt-oss-120b"`** — Groq's hosted version of OpenAI's open-source 120B model
- **`messages`** — standard chat format with a system prompt (role definition) and user message (the text to extract from)
- **`response_format`** — this is what enforces structured output:
  - `type: "json_schema"` tells the model to emit JSON conforming to a schema
  - `schema: CarListResponse.model_json_schema()` provides the exact schema generated from our Pydantic model

The LLM is constrained at decoding time to only produce tokens that result in valid JSON matching this schema.

In [10]:
system_prompt = (
    "You are an information extraction assistant. "
    "Given raw text from a webpage about electric cars, extract every electric vehicle "
    "mentioned along with its model name, price, and driving range in kilometres. "
    "If the range is not explicitly listed for a car, set range_kms to null."
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "car_list_response",
            "schema": CarListResponse.model_json_schema(),
        },
    },
)

raw_content = response.choices[0].message.content
print(raw_content[:800])

{"cars":[{"car_model":"Tata Nexon EV","price":"Rs. 14.49 Lakh","range_kms":465},{"car_model":"Tata Punch EV","price":"Rs. 10.98 Lakh","range_kms":421},{"car_model":"Tata Tiago EV","price":"Rs. 7.99 Lakh","range_kms":315},{"car_model":"Kia EV6","price":"Rs. 60.95 Lakh","range_kms":708},{"car_model":"MG Comet EV","price":"Rs. 6.98 Lakh","range_kms":230},{"car_model":"BYD Seal","price":"Rs. 41.00 Lakh","range_kms":650},{"car_model":"BMW i7","price":"Rs. 2.13 Crore","range_kms":625},{"car_model":"Mahindra XUV400 EV","price":"Rs. 15.49 Lakh","range_kms":456},{"car_model":"MG ZS EV","price":"Rs. 18.98 Lakh","range_kms":461},{"car_model":"Hyundai Kona Electric","price":"Rs. 23.84 Lakh","range_kms":452},{"car_model":"BMW iX","price":"Rs. 1.21 Crore","range_kms":425},{"car_model":"Rolls-Royce Spect


## Step 5: Parse and Validate the Response with Pydantic

The model returns a JSON **string**. We:

1. Parse it into a Python dict with `json.loads()`
2. Pass it to `CarListResponse.model_validate()` to get a typed Pydantic object

If the LLM returned malformed data — wrong field types, missing required fields, etc. — `model_validate` would raise a `ValidationError` and we'd know immediately. With JSON schema mode this almost never happens, but it's good defence.

In [11]:
car_list = CarListResponse.model_validate(json.loads(raw_content))

print(f"Extracted {len(car_list.cars)} cars\n")
print(json.dumps(car_list.model_dump(), indent=2)[:800])

Extracted 38 cars

{
  "cars": [
    {
      "car_model": "Tata Nexon EV",
      "price": "Rs. 14.49 Lakh",
      "range_kms": 465
    },
    {
      "car_model": "Tata Punch EV",
      "price": "Rs. 10.98 Lakh",
      "range_kms": 421
    },
    {
      "car_model": "Tata Tiago EV",
      "price": "Rs. 7.99 Lakh",
      "range_kms": 315
    },
    {
      "car_model": "Kia EV6",
      "price": "Rs. 60.95 Lakh",
      "range_kms": 708
    },
    {
      "car_model": "MG Comet EV",
      "price": "Rs. 6.98 Lakh",
      "range_kms": 230
    },
    {
      "car_model": "BYD Seal",
      "price": "Rs. 41.00 Lakh",
      "range_kms": 650
    },
    {
      "car_model": "BMW i7",
      "price": "Rs. 2.13 Crore",
      "range_kms": 625
    },
    {
      "car_model": "Mahindra XUV400 EV",
      "price": "Rs. 15.49 


Because `car_list` is a fully typed Pydantic object, we can access individual fields with autocomplete-friendly attribute syntax.

In [12]:
first_car = car_list.cars[0]
print(f"Model:  {first_car.car_model}")
print(f"Price:  {first_car.price}")
print(f"Range:  {first_car.range_kms} km")

Model:  Tata Nexon EV
Price:  Rs. 14.49 Lakh
Range:  465 km


## Step 6: Convert to a Pandas DataFrame

Pydantic's `model_dump()` converts the object back into plain Python dictionaries, which Pandas can ingest directly.

In [13]:
import pandas as pd

ev_df = pd.DataFrame([car.model_dump() for car in car_list.cars])
ev_df

,car_model,price,range_kms
0,Tata Nexon EV,Rs. 14.49 Lakh,465.0
1,Tata Punch EV,Rs. 10.98 Lakh,421.0
2,Tata Tiago EV,Rs. 7.99 Lakh,315.0
3,Kia EV6,Rs. 60.95 Lakh,708.0
4,MG Comet EV,Rs. 6.98 Lakh,230.0
5,BYD Seal,Rs. 41.00 Lakh,650.0
6,BMW i7,Rs. 2.13 Crore,625.0
7,Mahindra XUV400 EV,Rs. 15.49 Lakh,456.0
8,MG ZS EV,Rs. 18.98 Lakh,461.0
9,Hyundai Kona Electric,Rs. 23.84 Lakh,452.0


Save the result to CSV for downstream use.

In [15]:
ev_df.to_csv('ev_models.csv', index=False)
print("Saved to ev_models.csv")

Saved to ev_models.csv


## Summary & Key Takeaways

We built an end-to-end **HTML → structured data** pipeline in just a few cells:

| Step | Tool | Purpose |
|------|------|---------|
| 1 | BeautifulSoup | Strip HTML tags, extract visible text |
| 2 | Groq client | Connect to the GPT-OSS model |
| 3 | Pydantic | Declare a typed schema for the output |
| 4 | `response_format=json_schema` | Force the model to emit schema-conformant JSON |
| 5 | `model_validate` | Parse + validate into a typed object |
| 6 | Pandas | Tabular view, export to CSV |

### Why This Pattern Matters

- **No prompt engineering for output format.** The schema replaces fragile "return JSON like this..." instructions.
- **Schema lives in your code.** When you change a Pydantic model, the LLM contract updates automatically.
- **Composable.** Nest models, add validators, mark fields optional — all without changing the call to the LLM.

### Try It Yourself

- Add a `charging_time` field to `CarDetails` and re-run.
- Add a nested model for `BatteryDetails` (e.g. `kWh`, `chemistry`).
- Swap the model to `openai/gpt-oss-20b` and compare the speed/quality tradeoff.